# SupplyGraph: Static Homogeneous Graph Construction & Validation

**Author/Project**: Supply Chain Disruption & Risk Propagation Modeling  
**Dataset Source**: [SupplyGraph Benchmark Dataset (ciol-researchlab/SupplyGraph)](https://github.com/ciol-researchlab/SupplyGraph)  
**Raw Data Folder**: `Raw Dataset/Homogenoeus`  

## Pipeline Stages:
1. **Step 1: Data Inspection & Quality Audit** — Inspect all node, edge, and temporal files; verify counts; document duplication.
2. **Step 2: Node Feature Engineering** — Aggregate 8 temporal series per product into 24 static summary features (mean, std, linear trend slope).
3. **Step 3: Edge Construction** — Build an undirected NetworkX graph collapsing 4 relation types into a single integer-weighted edge (1 to 4).
4. **Step 4: Graph Validation** — Run structural and range sanity checks (degrees, isolated nodes, components, edge weights, NaNs).
5. **Step 5: PyG Conversion** — Convert the validated NetworkX graph to a PyTorch Geometric `Data` object (`torch_geometric.utils.from_networkx`).

---
### Data Quality Notice (POP001L12P Duplication):
> **Documented for Project Writeup:**  
> During Step 1 data inspection, `POP001L12P` appears twice in `Nodes.csv` (rows 12 and 20) and both rows are mapped to index `12` in `NodesIndex.csv`. The edge files exclusively connect **40 unique products** (indices `0` to `39`) yielding exactly **684 unique undirected edges**. In the temporal files, the two `POP001L12P` columns have distinct daily series.  
> **Resolution:** We adopt an unambiguous **40-node graph** (indices 0 to 39). For `POP001L12P`, we take the **average** of the two daily series across all 8 temporal features to maintain scale parity with the other 39 products without artificial inflation.

In [1]:
# Core Dependencies
import os
import sys
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from scipy.stats import linregress
import networkx as nx
import torch
import torch_geometric
from torch_geometric.utils import from_networkx

# Path Configuration
BASE_DIR = os.getcwd()
RAW_DIR = os.path.join(BASE_DIR, "Raw Dataset", "Homogenoeus")
print(f"Raw data directory: {RAW_DIR}")
print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | PyG: {torch_geometric.__version__}")

Raw data directory: C:\Users\91868\.gemini\antigravity\scratch\SupplyGraph\Raw Dataset\Homogenoeus
Python: 3.10.6 | PyTorch: 2.5.1+cu121 | PyG: 2.8.0.post1


## Step 1: Data Inspection

Load every relevant CSV file from `Raw Dataset/Homogenoeus` into pandas, display shapes, column schemas, data types, and samples, and verify node and edge counts.

In [2]:
# 1.1 Inspect Node Metadata Files
nodes_path = os.path.join(RAW_DIR, "Nodes", "Nodes.csv")
df_nodes = pd.read_csv(nodes_path)
print(f"=== Nodes.csv === | Shape: {df_nodes.shape}")
display(df_nodes.head(5))

nodes_idx_path = os.path.join(RAW_DIR, "Nodes", "NodesIndex.csv")
df_nodes_idx = pd.read_csv(nodes_idx_path)
print(f"=== NodesIndex.csv === | Shape: {df_nodes_idx.shape}")
display(df_nodes_idx.head(5))

nt_group_path = os.path.join(RAW_DIR, "Nodes", "Node Types (Product Group and Subgroup).csv")
df_nt_group = pd.read_csv(nt_group_path)
print(f"=== Node Types (Group/Subgroup) === | Shape: {df_nt_group.shape}")
display(df_nt_group.head(5))

nt_plant_path = os.path.join(RAW_DIR, "Nodes", "Nodes Type (Plant & Storage).csv")
df_nt_plant = pd.read_csv(nt_plant_path)
print(f"=== Nodes Type (Plant & Storage) === | Shape: {df_nt_plant.shape}")
display(df_nt_plant.head(5))

=== Nodes.csv === | Shape: (41, 1)
         Node
0  SOS008L02P
1  SOS005L04P
2  SOS003L04P
3  SOS002L09P
4  SOS001L12P
=== NodesIndex.csv === | Shape: (41, 2)
         Node  NodeIndex
0  SOS008L02P          0
1  SOS005L04P          1
2  SOS003L04P          2
3  SOS002L09P          3
4  SOS001L12P          4
=== Node Types (Group/Subgroup) === | Shape: (41, 3)
         Node Group Sub-Group
0  SOS008L02P     S       SOS
1  SOS005L04P     S       SOS
2  SOS003L04P     S       SOS
3  SOS002L09P     S       SOS
4  SOS001L12P     S       SOS
=== Nodes Type (Plant & Storage) === | Shape: (6545, 3)
         Node  Plant  Storage Location
0  POV002L09P   2120            2030.0
1  SOS001L12P   2120            2030.0
2  POP002L09P   2120            2030.0
3  POP001L12P   2120            2030.0
4  POP001L12P   2120            2030.0


In [3]:
# 1.2 Inspect Edge Files across all 4 relation categories
edge_files = [
    "Edges (Plant).csv",
    "Edges (Product Group).csv",
    "Edges (Product Sub-Group).csv",
    "Edges (Storage Location).csv"
]
edges_raw = {}
print("=== RAW EDGE FILES ===")
for ef in edge_files:
    p = os.path.join(RAW_DIR, "Edges", ef)
    df_e = pd.read_csv(p)
    edges_raw[ef] = df_e
    print(f"Edges/{ef:<30} -> Shape: {df_e.shape} | Columns: {list(df_e.columns)}")
    display(df_e.head(2))

print("\n=== EDGESINDEX FILES ===")
for ef in edge_files:
    p = os.path.join(RAW_DIR, "Edges", "EdgesIndex", ef)
    df_ei = pd.read_csv(p)
    print(f"EdgesIndex/{ef:<25} -> Shape: {df_ei.shape} | Columns: {list(df_ei.columns)}")

=== RAW EDGE FILES ===
Edges/Edges (Plant).csv              -> Shape: (1647, 3) | Columns: ['Plant', 'node1', 'node2']
   Plant         node1      node2
0   1901  ATWWP001K24P  ATN01K24P
1   1903        AT5X5K  ATN01K24P
Edges/Edges (Product Group).csv      -> Shape: (188, 3) | Columns: ['node1', 'node2', 'GroupCode']
        node1       node2 GroupCode
0  SOS008L02P  SOS005L04P         S
1  SOS008L02P  SOS003L04P         S
Edges/Edges (Product Sub-Group).csv  -> Shape: (52, 3) | Columns: ['node1', 'node2', 'SubGroupCode']
        node1       node2 SubGroupCode
0  SOS008L02P  SOS005L04P          SOS
1  SOS008L02P  SOS003L04P          SOS
Edges/Edges (Storage Location).csv   -> Shape: (3046, 3) | Columns: ['Storage Location', 'node1', 'node2']
   Storage Location         node1       node2
0             130.0  ATWWP001K24P   ATN01K24P
1             330.0    SOS005L04P  SOS002L09P

=== EDGESINDEX FILES ===
EdgesIndex/Edges (Plant).csv         -> Shape: (1647, 3) | Columns: ['Plant', 'node

In [4]:
# 1.3 Inspect Temporal Daily Feature Files (Unit and Weight)
temporal_manifest = {
    ("Production", "Unit"): ("Unit", "Production .csv"),
    ("Production", "Weight"): ("Weight", "Production .csv"),
    ("Sales_Order", "Unit"): ("Unit", "Sales Order.csv"),
    ("Sales_Order", "Weight"): ("Weight", "Sales Order .csv"),
    ("Delivery", "Unit"): ("Unit", "Delivery To distributor.csv"),
    ("Delivery", "Weight"): ("Weight", "Delivery to Distributor.csv"),
    ("Factory_Issue", "Unit"): ("Unit", "Factory Issue.csv"),
    ("Factory_Issue", "Weight"): ("Weight", "Factory Issue.csv"),
}

temporal_dfs = {}
print("=== TEMPORAL FILES SUMMARY ===")
for (feat, measure), (subdir, fname) in temporal_manifest.items():
    p = os.path.join(RAW_DIR, "Temporal Data", subdir, fname)
    df_t = pd.read_csv(p)
    temporal_dfs[(feat, measure)] = df_t
    print(f"{subdir}/{fname:<30} | Shape: {df_t.shape} | Dates: {df_t['Date'].iloc[0]} to {df_t['Date'].iloc[-1]} | Product cols: {df_t.shape[1]-1}")

display(temporal_dfs[("Production", "Unit")].iloc[:3, :6])

=== TEMPORAL FILES SUMMARY ===
Unit/Production .csv                | Shape: (221, 42) | Dates: 2023-01-01 00:00:00 to 2023-08-09 00:00:00 | Product cols: 41
Weight/Production .csv                | Shape: (221, 42) | Dates: 2023-01-01 00:00:00 to 2023-08-09 00:00:00 | Product cols: 41
Unit/Sales Order.csv                | Shape: (221, 42) | Dates: 2023-01-01 00:00:00 to 2023-08-09 00:00:00 | Product cols: 41
Weight/Sales Order .csv               | Shape: (221, 42) | Dates: 2023-01-01 00:00:00 to 2023-08-09 00:00:00 | Product cols: 41
Unit/Delivery To distributor.csv    | Shape: (221, 42) | Dates: 2023-01-01 00:00:00 to 2023-08-09 00:00:00 | Product cols: 41
Weight/Delivery to Distributor.csv    | Shape: (221, 42) | Dates: 2023-01-01 00:00:00 to 2023-08-09 00:00:00 | Product cols: 41
Unit/Factory Issue.csv              | Shape: (221, 42) | Dates: 2023-01-01 00:00:00 to 2023-08-09 00:00:00 | Product cols: 41
Weight/Factory Issue.csv              | Shape: (221, 42) | Dates: 2023-01-01 00:0

In [5]:
# 1.4 Count Verification and Quality Audit
num_rows_nodes = len(df_nodes)
unique_node_names = df_nodes["Node"].unique()
num_unique_nodes = len(unique_node_names)

print(f"Nodes.csv total rows          : {num_rows_nodes}")
print(f"Nodes.csv unique product names: {num_unique_nodes}")

# Compute unique undirected edges across all 4 relation files
pair_relations = defaultdict(set)
for rel_name, df_e in edges_raw.items():
    for _, row in df_e.iterrows():
        u = str(row["node1"])
        v = str(row["node2"])
        if u != v:
            pair = tuple(sorted([u, v]))
            pair_relations[pair].add(rel_name)

unique_edge_count = len(pair_relations)
weight_dist = Counter(len(rels) for rels in pair_relations.values())

print(f"Unique undirected edges across all 4 files: {unique_edge_count}")
print("Edge weight breakdown:")
for w in sorted(weight_dist.keys()):
    print(f"  Weight {w} (shared relations): {weight_dist[w]} edges")

print("\n--- AUDIT VERDICT ---")
print(f"Edge Count: 684 expected vs {unique_edge_count} actual -> EXACT MATCH.")
print(f"Node Count: 41 rows in file, but product 'POP001L12P' is duplicated (rows 12 and 20).")
print(f"            NodesIndex maps both to 12. EdgesIndex spans indices 0..39 (40 unique nodes).")
print(f"            -> Proceeding with 40-node graph representation.")

Nodes.csv total rows          : 41
Nodes.csv unique product names: 40
Unique undirected edges across all 4 files: 684
Edge weight breakdown:
  Weight 1 (shared relations): 297 edges
  Weight 2 (shared relations): 251 edges
  Weight 3 (shared relations): 91 edges
  Weight 4 (shared relations): 45 edges

--- AUDIT VERDICT ---
Edge Count: 684 expected vs 684 actual -> EXACT MATCH.
Node Count: 41 rows in file, but product 'POP001L12P' is duplicated (rows 12 and 20).
            NodesIndex maps both to 12. EdgesIndex spans indices 0..39 (40 unique nodes).
            -> Proceeding with 40-node graph representation.


## Step 2: Node Feature Engineering

For each of the 40 products, aggregate the 8 temporal features into **STATIC** summary statistics:
- **Mean** ($\mu$)
- **Standard Deviation** ($\sigma$)
- **Trend Slope** ($\beta_1$ from linear regression $y = \beta_1 t + \beta_0$)

Total features per node: $8 \times 3 = 24$ static features.  
*Note:* For product `POP001L12P`, the two distinct time series are averaged first.

In [6]:
# Ordered 40 products matching NodesIndex 0..39
df_unique = df_nodes_idx.drop_duplicates(subset=["Node"]).reset_index(drop=True)
ordered_products = df_unique["Node"].tolist()
print(f"Engineering features for {len(ordered_products)} products...")

series_keys = [
    ("Production", "Unit"),
    ("Production", "Weight"),
    ("Sales_Order", "Unit"),
    ("Sales_Order", "Weight"),
    ("Delivery", "Unit"),
    ("Delivery", "Weight"),
    ("Factory_Issue", "Unit"),
    ("Factory_Issue", "Weight"),
]

engineered_records = {prod: {} for prod in ordered_products}

for (feat_name, measure) in series_keys:
    df_t = temporal_dfs[(feat_name, measure)]
    prefix = f"{feat_name}_{measure.lower()}"
    num_days = len(df_t)
    time_steps = np.arange(num_days)

    # Average the two POP001L12P columns if both present
    if "POP001L12P" in df_t.columns and "POP001L12P.1" in df_t.columns:
        s_pop_avg = (df_t["POP001L12P"].astype(float) + df_t["POP001L12P.1"].astype(float)) / 2.0
    else:
        s_pop_avg = df_t["POP001L12P"].astype(float)

    for prod in ordered_products:
        if prod == "POP001L12P":
            values = s_pop_avg.values
        else:
            values = df_t[prod].astype(float).values

        # Mean, Std, Trend Slope
        mean_val = float(np.mean(values))
        std_val = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
        slope_res = linregress(time_steps, values)
        slope_val = float(slope_res.slope)

        engineered_records[prod][f"{prefix}_mean"] = mean_val
        engineered_records[prod][f"{prefix}_std"] = std_val
        engineered_records[prod][f"{prefix}_slope"] = slope_val

df_features = pd.DataFrame.from_dict(engineered_records, orient="index")
df_features.index.name = "Node"

print(f"\nEngineered Node Features DataFrame Shape: {df_features.shape}")
print(f"NaN values: {df_features.isna().sum().sum()} | Inf values: {np.isinf(df_features.values).sum()}")
display(df_features.head(5))

print("\nEngineered features for resolved product POP001L12P (first 6 cols):")
display(df_features.loc[["POP001L12P"], df_features.columns[:6]])

Engineering features for 40 products...

Engineered Node Features DataFrame Shape: (40, 24)
NaN values: 0 | Inf values: 0
            Production_unit_mean  Production_unit_std  Production_unit_slope  Production_weight_mean  Production_weight_std  Production_weight_slope  Sales_Order_unit_mean  Sales_Order_unit_std  Sales_Order_unit_slope  Sales_Order_weight_mean  Sales_Order_weight_std  Sales_Order_weight_slope  Delivery_unit_mean  Delivery_unit_std  Delivery_unit_slope  Delivery_weight_mean  Delivery_weight_std  Delivery_weight_slope  Factory_Issue_unit_mean  Factory_Issue_unit_std  Factory_Issue_unit_slope  Factory_Issue_weight_mean  Factory_Issue_weight_std  Factory_Issue_weight_slope
Node                                                                                                                                                                                                                                                                                                           

## Step 3: Edge Construction

Build a NetworkX undirected graph:
- Connect products that share a Plant, Product Group, Product Sub-Group, or Storage Location.
- Collapse relations into a single weighted edge per product pair ($weight \in \{1, 2, 3, 4\}$).
- Attach the 24 engineered features as vector attribute `x` on each node.

In [7]:
prod_to_idx = {prod: i for i, prod in enumerate(ordered_products)}
idx_to_prod = {i: prod for i, prod in enumerate(ordered_products)}
num_nodes = len(ordered_products)
feature_cols = list(df_features.columns)

G = nx.Graph()

# Add nodes with metadata and features
for i, prod in enumerate(ordered_products):
    node_feat_vector = [float(df_features.loc[prod, c]) for c in feature_cols]
    G.add_node(
        i,
        node_id=prod,
        node_index=i,
        x=node_feat_vector
    )

# Collapse multi-relational edges into weighted edges
relation_tracker = defaultdict(lambda: {"count": 0, "relations": set()})
relation_names = {
    "Edges (Plant).csv": "Plant",
    "Edges (Product Group).csv": "Group",
    "Edges (Product Sub-Group).csv": "SubGroup",
    "Edges (Storage Location).csv": "Storage"
}

for ef, rel_label in relation_names.items():
    df_e = edges_raw[ef]
    for _, row in df_e.iterrows():
        u_name = str(row["node1"])
        v_name = str(row["node2"])
        if u_name != v_name and u_name in prod_to_idx and v_name in prod_to_idx:
            u = prod_to_idx[u_name]
            v = prod_to_idx[v_name]
            pair = tuple(sorted([u, v]))
            relation_tracker[pair]["count"] += 1
            relation_tracker[pair]["relations"].add(rel_label)

# Attach weighted edges
for (u, v), info in relation_tracker.items():
    weight = float(len(info["relations"]))
    G.add_edge(u, v, weight=weight)

print(f"NetworkX Graph Construction:")
print(f"  Total nodes: {G.number_of_nodes()}")
print(f"  Total edges: {G.number_of_edges()} (Expected: 684) -> {'MATCH' if G.number_of_edges() == 684 else 'MISMATCH'}")

sample_edges = list(G.edges(data=True))[:5]
print("\nSample edges:")
for u, v, d in sample_edges:
    print(f"  ({idx_to_prod[u]:<12}, {idx_to_prod[v]:<12}) -> Weight: {d['weight']}")

NetworkX Graph Construction:
  Total nodes: 40
  Total edges: 684 (Expected: 684) -> MATCH

Sample edges:
  (SOS008L02P  , SOS005L04P  ) -> Weight: 4.0
  (SOS008L02P  , SOS002L09P  ) -> Weight: 4.0
  (SOS008L02P  , SOS003L04P  ) -> Weight: 4.0
  (SOS008L02P  , SOS500M24P  ) -> Weight: 4.0
  (SOS008L02P  , SOS001L12P  ) -> Weight: 4.0


## Step 4: Validation

Run thorough structural and sanity checks before PyG conversion:
1. Node and edge count checks
2. Isolated nodes check
3. Degree distribution analysis (scale-free vs uniform dense analysis)
4. Connected components analysis
5. Edge weights sanity range checks
6. Node feature matrix sanity checks (no NaNs, no Infs, non-negative variances)

In [8]:
# 4.1 Node & Edge Counts
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
assert num_nodes == 40, f"Expected 40 nodes, got {num_nodes}"
assert num_edges == 684, f"Expected 684 edges, got {num_edges}"
print(f"1. Node Count: {num_nodes} [PASS]")
print(f"2. Edge Count: {num_edges} [PASS]")

# 4.2 Isolated Nodes
isolated_nodes = [n for n, d in G.degree() if d == 0]
assert len(isolated_nodes) == 0, f"Isolated nodes found: {isolated_nodes}"
print(f"3. Isolated Nodes (degree = 0): {len(isolated_nodes)} [PASS]")

# 4.3 Degree Distribution
degrees = [d for _, d in G.degree()]
density = nx.density(G)
print(f"\n4. Degree Distribution Analysis:")
print(f"   Min Degree   : {np.min(degrees)}")
print(f"   Max Degree   : {np.max(degrees)} (Theoretical max = {num_nodes - 1})")
print(f"   Mean Degree  : {np.mean(degrees):.2f}")
print(f"   Median Degree: {np.median(degrees):.1f}")
print(f"   Std Dev      : {np.std(degrees):.2f}")
print(f"   Graph Density: {density * 100:.2f}%")
print("   [Topology Note] Degree distribution is dense and uniform (not heavy-tailed/scale-free)")
print("   due to central sharing of storage locations and manufacturing plants across FMCG SKUs.")

# 4.4 Connected Components
num_cc = nx.number_connected_components(G)
cc_sizes = [len(c) for c in nx.connected_components(G)]
assert num_cc == 1, f"Expected 1 connected component, got {num_cc}"
print(f"\n5. Connected Components: {num_cc} (Sizes: {cc_sizes}) [PASS]")

# 4.5 Edge Weights Sanity Check
weights = [data["weight"] for _, _, data in G.edges(data=True)]
valid_weights = all(w in (1.0, 2.0, 3.0, 4.0) for w in weights)
assert valid_weights, "Invalid edge weights detected"
print(f"\n6. Edge Weights Range: Min = {min(weights)}, Max = {max(weights)} [PASS]")

# 4.6 Node Feature Range Checks
nan_count = np.isnan(df_features.values).sum()
inf_count = np.isinf(df_features.values).sum()
std_cols = [c for c in df_features.columns if c.endswith("_std")]
neg_stds = (df_features[std_cols] < 0).sum().sum()
mean_cols = [c for c in df_features.columns if c.endswith("_mean")]
neg_means = (df_features[mean_cols] < 0).sum().sum()

assert nan_count == 0, "NaNs found in features"
assert inf_count == 0, "Infs found in features"
assert neg_stds == 0, "Negative standard deviations found"
assert neg_means == 0, "Negative mean volumes found"
print(f"\n7. Feature Sanity Checks (NaNs: 0, Infs: 0, Neg Stds: 0, Neg Means: 0) [PASS]")
print("\n>>> ALL VALIDATION CHECKS PASSED SUCCESSFULLY!")

1. Node Count: 40 [PASS]
2. Edge Count: 684 [PASS]
3. Isolated Nodes (degree = 0): 0 [PASS]

4. Degree Distribution Analysis:
   Min Degree   : 25
   Max Degree   : 39 (Theoretical max = 39)
   Mean Degree  : 34.20
   Median Degree: 37.0
   Std Dev      : 5.21
   Graph Density: 87.69%
   [Topology Note] Degree distribution is dense and uniform (not heavy-tailed/scale-free)
   due to central sharing of storage locations and manufacturing plants across FMCG SKUs.

5. Connected Components: 1 (Sizes: [40]) [PASS]

6. Edge Weights Range: Min = 1.0, Max = 4.0 [PASS]

7. Feature Sanity Checks (NaNs: 0, Infs: 0, Neg Stds: 0, Neg Means: 0) [PASS]

>>> ALL VALIDATION CHECKS PASSED SUCCESSFULLY!


## Step 5: Convert to PyTorch Geometric (PyG)

Convert the validated NetworkX graph to a `torch_geometric.data.Data` object using `from_networkx`:
- `x`: Node feature matrix of shape `[40, 24]`
- `edge_index`: Graph edge indices of shape `[2, 1368]` (bidirectional undirected edges)
- `edge_weight`: Edge weights of shape `[1368]`

In [9]:
# Convert via torch_geometric.utils.from_networkx
pyg_data = from_networkx(
    G,
    group_node_attrs=["x"],
    group_edge_attrs=["weight"]
)

# Cast to PyTorch float32 tensors
pyg_data.x = pyg_data.x.to(torch.float32)
pyg_data.edge_weight = pyg_data.edge_attr.view(-1).to(torch.float32)

print("=== PyTorch Geometric Data Object ===")
print(pyg_data)
print(f"  data.x shape          : {list(pyg_data.x.shape)} | dtype: {pyg_data.x.dtype}")
print(f"  data.edge_index shape : {list(pyg_data.edge_index.shape)} | dtype: {pyg_data.edge_index.dtype}")
print(f"  data.edge_weight shape: {list(pyg_data.edge_weight.shape)} | dtype: {pyg_data.edge_weight.dtype}")
print(f"  Number of nodes       : {pyg_data.num_nodes}")
print(f"  Number of edges       : {pyg_data.num_edges}")
print(f"  Is undirected         : {pyg_data.is_undirected()}")
print(f"  Has isolated nodes    : {pyg_data.has_isolated_nodes()}")
print(f"  Has self loops        : {pyg_data.has_self_loops()}")

# Verification assertions
assert pyg_data.num_nodes == 40
assert pyg_data.edge_index.shape == (2, 1368)
assert pyg_data.x.shape == (40, 24)
assert pyg_data.edge_weight.shape == (1368,)
print("\n>>> SUCCESS: Validated static graph object built and ready for GNN modeling!")

=== PyTorch Geometric Data Object ===
Data(edge_index=[2, 1368], node_id=[40], node_index=[40], x=[40, 24], edge_attr=[1368, 1], edge_weight=[1368])
  data.x shape          : [40, 24] | dtype: torch.float32
  data.edge_index shape : [2, 1368] | dtype: torch.int64
  data.edge_weight shape: [1368] | dtype: torch.float32
  Number of nodes       : 40
  Number of edges       : 1368
  Is undirected         : True
  Has isolated nodes    : False
  Has self loops        : False

>>> SUCCESS: Validated static graph object built and ready for GNN modeling!
